In [ ]:
# RNNによる文章生成
# 言語モデルを使った文章生成
# RNNによる文章生成の手順
# 文章生成の実装
import sys
sys.path.append('../..')
import numpy as np
from common.functions import softmax
from ch06.rnnlm import Rnnlm
from ch06.better_rnnlm import BetterRnnlm


class RnnlmGen(Rnnlm):
    # start_id：文章の最初の単語ID
    # skip_ids：生成をスキップする単語IDのリスト
    # sample_size：生成する単語の総数
    def generate(self, start_id, skip_ids=None, sample_size=100):
        # word_ids：生成された単語IDを順番に記録していくためのリスト
        # 最初はstart_idしか渡されないので、start_idをリストに追加
        word_ids = [start_id]

        # 次のRNNモデルに入力するための変数xを作成
        # 最初はstart_id ➡ ループ中で新しく生成された単語に上書きされていく
        x = start_id
        while len(word_ids) < sample_size:  # リストの長さ（生成された単語数）がsample_sizeの値になるまで繰り返す
            # 今持ってる入力単語をNumpy配列に変換し、1行1列に変形
            x = np.array(x).reshape(1, 1)
            # 第6章で作成したBetterRnnlmのpredict
            score = self.predict(x)
            # ソフトマックス関数を通して確率分布を求める
            # flatten()：1次元配列に変換（softmax関数は1次元配列の入力を前提として作られているため）
            p = softmax(score.flatten())

            # 確率分布pから1つの単語をサンプリング
            sampled = np.random.choice(len(p), size=1, p=p)
            # サンプリングした単語がskip_idsに含まれていない場合、
            if (skip_ids is None) or (sampled not in skip_ids):
                x = sampled  # サンプリングした値を新しい入力単語へ
                word_ids.append(int(x))  # 生成された単語を追加

        return word_ids

    # 現在のLSTM層が憶えている隠れ状態hと記憶セルcを取得
    def get_state(self):
        return self.lstm_layer.h, self.lstm_layer.c

    # 外から保存された記憶（state）を受け取ってLSTMにセット
    def set_state(self, state):
        self.lstm_layer.set_state(*state)
